# Ground Motion Analysis and Validation

**Notebook:** 02_ground_motions.ipynb

**Purpose:** Load, analyze, and validate ground motion records for seismic drift research

**Contents:**
1. Load GM records from PEER NGA format
2. Extract and compute intensity measures (PGA, PGV, Sa)
3. Validate records against BNBC 2020 parameters
4. Visualize acceleration histories and spectra
5. Export validated records for IDA analysis

## 1. Environment Setup and Imports

In [ ]:
# Import standard libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import logging

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('notebook')

# Add project to path
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

logger.info(f'Project root: {project_root}')
logger.info(f'Python version: {sys.version}')

In [ ]:
# Import project modules
from src.ida.gm_loader import GMRecord, load_ground_motion, generate_synthetic_gm
from src.ida.gm_scaler import GMScaler, compute_spectrum

# Load BNBC parameters
with open(project_root / 'config' / 'bnbc_parameters.yaml') as f:
    bnbc_params = yaml.safe_load(f)

with open(project_root / 'config' / 'analysis_config.yaml') as f:
    analysis_config = yaml.safe_load(f)

logger.info('✓ All imports successful')
print(f'BNBC Zones: {list(bnbc_params["seismic_zones"].keys())}')
print(f'IDA Configuration: {analysis_config["ida"]}')

## 2. Load and Validate Ground Motions

In [ ]:
# Generate synthetic ground motions for demonstration
# (In production, these would be loaded from PEER NGA database)

synthetic_gms = {}
zones = [1, 2, 3, 4]

for zone in zones:
    zone_key = f'zone_{zone}'
    pga_target = bnbc_params['seismic_zones'][zone_key]['pga']
    
    # Generate 3 synthetic records per zone
    synthetic_gms[zone_key] = []
    for i in range(3):
        gm = generate_synthetic_gm(
            duration=30.0,
            dt=0.01,
            n_modes=10,
            pga=pga_target * (0.8 + 0.4 * np.random.rand()),
            seed=42 + zone * 10 + i
        )
        gm.name = f'synthetic_{zone_key}_{i}'
        synthetic_gms[zone_key].append(gm)

print(f'Generated {sum(len(v) for v in synthetic_gms.values())} synthetic ground motions')
print(f'Distribution: {[(k, len(v)) for k, v in synthetic_gms.items()]}')

In [ ]:
# Analyze ground motion properties
gm_summary = []

for zone_key, gm_list in synthetic_gms.items():
    for gm in gm_list:
        gm_summary.append({
            'Zone': zone_key,
            'Name': gm.name,
            'PGA (g)': gm.pga,
            'PGV (cm/s)': gm.pgv,
            'PGD (cm)': gm.pgd,
            'Duration (s)': gm.duration,
            'Cycles': gm.cycles,
            'dt (s)': gm.dt
        })

df_gm = pd.DataFrame(gm_summary)
print('\nGround Motion Summary:')
print(df_gm.to_string())

print('\n\nSummary Statistics by Zone:')
print(df_gm.groupby('Zone')[['PGA (g)', 'PGV (cm/s)', 'PGD (cm)']].describe().round(3))

## 3. Scale Ground Motions to Target Intensity

In [ ]:
# Scale ground motions for IDA analysis
target_period = analysis_config['ida']['target_period']  # Usually T1 of building
damping = analysis_config['ida']['damping']  # 5% damping
sa_range = np.arange(
    analysis_config['ida']['sa_range'][0],
    analysis_config['ida']['sa_range'][1] + analysis_config['ida']['sa_step'],
    analysis_config['ida']['sa_step']
)

print(f'Target Period: {target_period:.2f} s')
print(f'Damping: {damping}')
print(f'Sa Range: {sa_range[0]:.2f}g to {sa_range[-1]:.2f}g')
print(f'Number of intensity levels: {len(sa_range)}')

In [ ]:
# Test scaling on Zone 3 record (Dhaka region)
test_gm = synthetic_gms['zone_3'][0]
print(f'Original GM: {test_gm.name}')
print(f'  Original PGA: {test_gm.pga:.3f} g')

# Create scaler
scaler = GMScaler(test_gm, target_period=target_period, damping=damping)

# Scale to different Sa values
scaled_gms = {}
for target_sa in sa_range[::4]:  # Sample every 4th level for display
    scaled_gm = scaler.scale_to_sa(target_sa, method='linear')
    scaled_gms[target_sa] = scaled_gm
    print(f'  Sa={target_sa:.2f}g → PGA={scaled_gm.pga:.3f}g')

print(f'\n✓ Successfully scaled {len(scaled_gms)} GM records')

In [ ]:
# Visualize acceleration histories
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, zone in enumerate(zones):
    zone_key = f'zone_{zone}'
    gm = synthetic_gms[zone_key][0]
    
    ax = axes[idx]
    ax.plot(gm.time, gm.acceleration, 'b-', linewidth=0.8, alpha=0.7)
    ax.axhline(gm.pga, color='r', linestyle='--', label=f'PGA={gm.pga:.3f}g')
    ax.axhline(-gm.pga, color='r', linestyle='--')
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Acceleration (g)', fontsize=10)
    ax.set_title(f'{zone_key.upper()}: {gm.name}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(project_root / 'results' / 'figures' / 'gm_acceleration_histories.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: gm_acceleration_histories.png')

## 4. Validate Records Against BNBC 2020 Parameters

In [ ]:
# Validate PGA compliance with BNBC zones
validation_results = []

for zone in zones:
    zone_key = f'zone_{zone}'
    bnbc_pga = bnbc_params['seismic_zones'][zone_key]['pga']
    
    for gm in synthetic_gms[zone_key]:
        # Allow ±50% variation for synthetic records
        pga_min = bnbc_pga * 0.5
        pga_max = bnbc_pga * 1.5
        
        validation_results.append({
            'Zone': zone_key,
            'GM': gm.name,
            'BNBC PGA (g)': bnbc_pga,
            'Actual PGA (g)': gm.pga,
            'PGA Min (g)': pga_min,
            'PGA Max (g)': pga_max,
            'Valid': pga_min <= gm.pga <= pga_max,
            'Duration (s)': gm.duration,
            'Duration Valid': gm.duration >= 10.0
        })

df_validation = pd.DataFrame(validation_results)
print('Ground Motion Validation Results:')
print(df_validation.to_string())

valid_count = df_validation['Valid'].sum()
duration_valid = df_validation['Duration Valid'].sum()
print(f'\n✓ {valid_count}/{len(df_validation)} records have valid PGA')
print(f'✓ {duration_valid}/{len(df_validation)} records have sufficient duration')

In [ ]:
# Visualize PGA comparison with BNBC zones
fig, ax = plt.subplots(figsize=(12, 6))

zone_labels = [f'Zone {z}' for z in zones]
bnbc_pgas = [bnbc_params['seismic_zones'][f'zone_{z}']['pga'] for z in zones]

# Plot BNBC PGA values
x_positions = np.arange(len(zones))
ax.bar(x_positions, bnbc_pgas, width=0.3, label='BNBC Design PGA', color='steelblue', alpha=0.7)

# Plot actual GM PGA values
for zone_idx, zone in enumerate(zones):
    zone_key = f'zone_{zone}'
    pgas = [gm.pga for gm in synthetic_gms[zone_key]]
    offsets = np.random.normal(0, 0.02, len(pgas))  # Add jitter
    ax.scatter([zone_idx + 0.3 + o for o in offsets], pgas, s=100, alpha=0.6, color='coral', marker='o')

ax.set_xlabel('Seismic Zone', fontsize=12, fontweight='bold')
ax.set_ylabel('Peak Ground Acceleration (g)', fontsize=12, fontweight='bold')
ax.set_title('Ground Motion PGA vs BNBC 2020 Design Parameters', fontsize=13, fontweight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels(zone_labels)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(project_root / 'results' / 'figures' / 'gm_pga_validation.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: gm_pga_validation.png')

## 5. Spectral Content Analysis

In [ ]:
# Compute response spectra for selected records
periods = np.logspace(-1, 1.5, 50)  # 0.1s to ~30s
spectra = {}

for zone in zones:
    zone_key = f'zone_{zone}'
    gm = synthetic_gms[zone_key][0]
    
    sa = compute_spectrum(gm, periods, damping=5)
    spectra[zone_key] = sa

# Plot response spectra for all zones
fig, ax = plt.subplots(figsize=(12, 7))

colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(zones)))

for idx, zone in enumerate(zones):
    zone_key = f'zone_{zone}'
    sa = spectra[zone_key]
    ax.loglog(periods, sa, marker='o', markersize=4, linewidth=2, 
              label=f'{zone_key.upper()} (PGA={bnbc_params["seismic_zones"][zone_key]["pga"]:.2f}g)', 
              color=colors[idx])

ax.set_xlabel('Period (s)', fontsize=12, fontweight='bold')
ax.set_ylabel('Spectral Acceleration Sa(T) (g)', fontsize=12, fontweight='bold')
ax.set_title('Response Spectra by BNBC Seismic Zone', fontsize=13, fontweight='bold')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig(project_root / 'results' / 'figures' / 'gm_response_spectra.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved: gm_response_spectra.png')

## 6. Summary and Export

In [ ]:
# Create comprehensive summary
print('='*60)
print('GROUND MOTION ANALYSIS SUMMARY')
print('='*60)

print(f'\nTotal Records Analyzed: {sum(len(v) for v in synthetic_gms.values())}')
print(f'\nRecords by Zone:')
for zone in zones:
    zone_key = f'zone_{zone}'
    count = len(synthetic_gms[zone_key])
    print(f'  {zone_key.upper()}: {count} records')

print(f'\nValidation Results:')
print(f'  PGA Compliance: {(df_validation["Valid"].sum() / len(df_validation) * 100):.1f}%')
print(f'  Duration Compliance: {(df_validation["Duration Valid"].sum() / len(df_validation) * 100):.1f}%')

print(f'\nIDA Configuration:')
print(f'  Target Period: {target_period:.2f} s')
print(f'  Sa Range: {sa_range[0]:.2f}g to {sa_range[-1]:.2f}g')
print(f'  Intensity Levels: {len(sa_range)}')
print(f'  Total IDA Runs: ~{len(sa_range) * sum(len(v) for v in synthetic_gms.values()) * 5} (5 buildings)')

print('\n' + '='*60)
print('✓ Ground motion analysis complete!')
print('='*60)

In [ ]:
# Save ground motion metadata to CSV
df_gm.to_csv(project_root / 'data' / 'metadata' / 'ground_motions.csv', index=False)
df_validation.to_csv(project_root / 'data' / 'metadata' / 'gm_validation_results.csv', index=False)

print('✓ Saved metadata:')
print(f'  - {project_root / "data" / "metadata" / "ground_motions.csv"}')
print(f'  - {project_root / "data" / "metadata" / "gm_validation_results.csv"}')